# Projeto 2 - Alegoria da Caverna

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 1400
largura = 1400

window = glfw.create_window(largura, altura, "Alegoria da Caverna", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [3]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [4]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []



def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    img = Image.open(img_textura).convert("RGB")
    
    img_width = img.size[0]
    img_height = img.size[1]
    
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            if texture_id > 0 and texture_id <= len(modelo['texture']):
                textures_coord_list.append(modelo['texture'][texture_id - 1])
            else: #Tratando o modelo que não tem textura
                textures_coord_list.append([0.0, 0.0])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    textureId = numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1

    return verticeInicial, verticeFinal - verticeInicial, textureId


In [5]:
def add_textured_plane_xz(x0, x1, z0, z1, y, repeat_u=1.0, repeat_v=1.0):
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    p1 = [x0, y, z0]
    p2 = [x1, y, z0]
    p3 = [x1, y, z1]
    p4 = [x0, y, z1]

    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)
    quantidade_vertices = vertice_final - vertice_inicial

    return vertice_inicial, quantidade_vertices


def add_floor_sky_textured(x0, x1, z0, z1, y, arq_textura, repeat_u=1.0, repeat_v=1.0):
    inicio, qtd = add_textured_plane_xz(
        x0, x1,
        z0, z1,
        y,
        repeat_u,
        repeat_v
    )

    global numberTextures
    floor_sky_texture = numberTextures

    load_texture_from_file(floor_sky_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, floor_sky_texture

In [6]:
def add_textured_plane_xy(x0, x1, y0, y1, z, repeat_u=1.0, repeat_v=1.0):
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    p1 = [x0, y0, z]
    p2 = [x1, y0, z]
    p3 = [x1, y1, z]
    p4 = [x0, y1, z]

    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)
    return vertice_inicial, vertice_final - vertice_inicial

def add_top_bottom_textured(x0, x1, y0, y1, z, arq_textura, repeat_u=1.0, repeat_v=1.0):
    inicio, qtd = add_textured_plane_xy(
        x0, x1,
        y0, y1,
        z,
        repeat_u,
        repeat_v
    )

    global numberTextures
    top_bottom_texture = numberTextures

    load_texture_from_file(top_bottom_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, top_bottom_texture

In [7]:
def add_textured_plane_yz(y0, y1, z0, z1, x, repeat_u=1.0, repeat_v=1.0):
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    p1 = [x, y0, z0]
    p2 = [x, y0, z1]
    p3 = [x, y1, z1]
    p4 = [x, y1, z0]

    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)
    return vertice_inicial, vertice_final - vertice_inicial

def add_right_left_textured(y0, y1, z0, z1, x, arq_textura, repeat_u=1.0, repeat_v=1.0):
    inicio, qtd = add_textured_plane_yz(
        y0, y1,
        z0, z1,
        x,
        repeat_u,
        repeat_v
    )

    global numberTextures
    right_left_texture = numberTextures

    load_texture_from_file(right_left_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, right_left_texture

### Vamos carregar cada modelo e definir funções para desenhá-los

#### Criando o chão externo da cena

O objeto da caverna (cena interna) já tem um chão e uma textura interna, então somente cria-se um chão para o cenário externo.

In [8]:
verticeInicial_areia, qtdVertices_areia, textureId_areia = add_floor_sky_textured(
    -15, 15, -15, 15, -1.0, "cenario/areia.jpg", 8.0, 8.0
)

def desenha_areia():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_areia)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_areia, qtdVertices_areia) ## renderizando

verticeInicial_mar, qtdVertices_mar, textureId_mar = add_floor_sky_textured(
     -30, 30, -30, 30, -1.05, "cenario/ceu/bottom.png", 1.0, 1.0
)

def desenha_mar():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_mar)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_mar, qtdVertices_mar) ## renderizando


xmin = -30
xmax =  30
zmin = -30
zmax =  30
ymin = -2
ymax =  28

verticeInicial_ceu, qtdVertices_ceu, textureId_ceu = add_floor_sky_textured(
    xmin, xmax, zmin, zmax, ymax, "cenario/ceu/top.png", 1.0, -1.0
)

def desenha_ceu_top():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_ceu)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ceu, qtdVertices_ceu) ## renderizando

verticeInicial_right, qtdVertices_right, textureId_right = add_right_left_textured(
    ymin, ymax, zmin, zmax, xmax, "cenario/ceu/right.png", -1.0, 1.0
)

def desenha_ceu_right():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_right)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_right, qtdVertices_right) ## renderizando

verticeInicial_left, qtdVertices_left, textureId_left = add_right_left_textured(
    ymin, ymax, zmin, zmax, xmin, "cenario/ceu/left.png", 1.0, 1.0
)

def desenha_ceu_left():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_left)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_left, qtdVertices_left) ## renderizando


verticeInicial_front, qtdVertices_front, textureId_front = add_top_bottom_textured(
    xmin, xmax, ymin, ymax, zmax, "cenario/ceu/front.png", 1.0, 1.0
)

def desenha_ceu_front():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_front)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_front, qtdVertices_front) ## renderizando

verticeInicial_back, qtdVertices_back, textureId_back = add_top_bottom_textured(
    xmin, xmax, ymin, ymax, zmin, "cenario/ceu/back.png", -1.0, 1.0
)

def desenha_ceu_back():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_back)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_back, qtdVertices_back) ## renderizando



0
1
2
3
4
5
6


In [9]:
# Carregando os objetos do cenário

verticeInicial_caverna, quantosVertices_caverna, textureId_caverna = load_obj_and_texture('objetos/caverna/GRUTA_BASE.obj', ['objetos/caverna/GRUTA_BASE_defaultMat_BaseColor.png'])
verticeInicial_muro, qtdVertices_muro, textureId_muro = load_obj_and_texture('objetos/muro/muro.obj', ['objetos/muro/muro.png'])
verticeInicial_montePedras,qtdVertices_montePedras, textureId_montePedras = load_obj_and_texture('objetos/pedra_circulo/pedra_circulo.obj', ['objetos/pedra_circulo/pedra_circulo.png']) 

global verticeInicial_fogueira, qtdVertices_fogueira

verticeInicial_fogueira, qtdVertices_fogueira, textureId_fogueira = load_obj_and_texture('objetos/fogueira/fogueira.obj', ['objetos/fogueira/Koster_Base_Color.png'])
verticeInicial_fire, qtdVertices_fire, textureId_fire = load_obj_and_texture('objetos/fire/fire.obj', ['objetos/fire/fire.png'])

verticeInicial_prisioneiro, qtdVertices_prisioneiro, textureId_prisioneiro = load_obj_and_texture('objetos/prisioneiro/prisioneiro.obj', ['objetos/prisioneiro/prisioneiro.png'])

def desenha_caverna(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_caverna)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_caverna, quantosVertices_caverna) ## renderizando

def desenha_muro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_muro)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_muro, qtdVertices_muro) ## renderizando
    
def desenha_montePedras(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId_montePedras)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_montePedras, qtdVertices_montePedras) ## renderizando

def desenha_fogueira(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_fogueira)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fogueira, qtdVertices_fogueira) ## renderizando

def desenha_fire(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    cx, cy, cz = centro_objeto(verticeInicial_fire, qtdVertices_fire)
    
    mat_model = model_com_centro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, cx, cy, cz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_fire)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fire, qtdVertices_fire) ## renderizando

def desenha_prisioneiro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)

    glBindTexture(GL_TEXTURE_2D, textureId_prisioneiro)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_prisioneiro, qtdVertices_prisioneiro) ## renderizando    

Processando modelo objetos/caverna/GRUTA_BASE.obj. Vertice inicial: 42
Processando modelo objetos/caverna/GRUTA_BASE.obj. Vertice final: 30456
7
Processando modelo objetos/muro/muro.obj. Vertice inicial: 30456
Processando modelo objetos/muro/muro.obj. Vertice final: 199560
8
Processando modelo objetos/pedra_circulo/pedra_circulo.obj. Vertice inicial: 199560
Processando modelo objetos/pedra_circulo/pedra_circulo.obj. Vertice final: 229527
9
Processando modelo objetos/fogueira/fogueira.obj. Vertice inicial: 229527
Processando modelo objetos/fogueira/fogueira.obj. Vertice final: 234957
10
Processando modelo objetos/fire/fire.obj. Vertice inicial: 234957
Processando modelo objetos/fire/fire.obj. Vertice final: 237441
11
Processando modelo objetos/prisioneiro/prisioneiro.obj. Vertice inicial: 237441
Processando modelo objetos/prisioneiro/prisioneiro.obj. Vertice final: 859557
12


### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [10]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [11]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [12]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [13]:
#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

# camera
cameraPos   = glm.vec3(1.5, 0.9, 8)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode, apagar_fogo

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if key==glfw.KEY_F and action == glfw.PRESS:
        apagar_fogo = not apagar_fogo 

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Funções

In [14]:
def centro_objeto(inicio, qtd):
    '''Função para calcular o centro de um objeto - usar como referência para as transformações geometricas'''
    xs, ys, zs = [], [], []

    for i in range(inicio, inicio + qtd):
        x, y, z = vertices_list[i]
        xs.append(float(x))
        ys.append(float(y))
        zs.append(float(z))

    cx = sum(xs) / len(xs)
    cy = sum(ys) / len(ys)
    cz = sum(zs) / len(zs)
    return cx, cy, cz

### Matrizes Model, View e Projection

In [15]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def model_com_centro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, c_x, c_y, c_z):
    angle = math.radians(angle)

    matrix_transform = glm.mat4(1.0)

    # leva para a posição final no mundo
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))

    # rotação no eixo desejado
    if angle != 0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))

    # traz o centro para a origem, escala, e devolve
    matrix_transform = glm.translate(matrix_transform, glm.vec3(c_x, c_y, c_z))
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    matrix_transform = glm.translate(matrix_transform, glm.vec3(-c_x, -c_y, -c_z))

    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Nesse momento, nós exibimos a janela!


In [16]:
glfw.show_window(window)

### Loop principal da janela.

In [17]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

#Controle do fogo
global apagar_fogo
apagar_fogo = False 

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)
    
    desenha_areia()
    desenha_mar()
    desenha_ceu_top()
    desenha_ceu_right()
    desenha_ceu_left()
    desenha_ceu_front()
    desenha_ceu_back()
    desenha_caverna(0.0, 0.0, 0, 1, 0.5, 1.4, -5, 2.5, 2, -2.5)

    desenha_muro(90, 0, 1, 0, 0.9, -0.92, 0, 1, 1, 1)

    add_y = 0
    for i in range(8):
        desenha_montePedras(0, 0, 1, 0, 2.8, -0.75+add_y, -4, 0.5, 0.5, 0.5)
        add_y += 0.1

    desenha_fogueira(0, 0, 0, 0, 7.1, 0, -4.4, 0.8, 0.8, 0.8)

    cx, cy, cz = centro_objeto(verticeInicial_fogueira, qtdVertices_fogueira)

    if apagar_fogo:
        sx_fire = 0
        sy_fire = 0
        sz_fire = 0
    else: #Efeito de pulsação (escala vai variar de acordo com o seno)
        sx_fire = 100 + 8 * math.sin(4 * glfw.get_time())
        sy_fire = 100
        sz_fire = 100 + 8 * math.sin(4 * glfw.get_time())
    
    desenha_fire(0, 0, 0, 0, cx+8.1, cy+0.2, cz-4.6, sx_fire, sy_fire, sz_fire)

    add_z = 0
    for i in range(4):
        desenha_prisioneiro(270, 0, 1, 0, -0.49, -0.86, 3+add_z, 1.1, 1.1, 1.1)
        add_z -= 1.5        
        
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)

glfw.terminate()